## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr # for Chat Interface

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [3]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

   
Contact
amitsachdeva45@gmail.com
www.linkedin.com/in/amit-
sachdeva-878a7b115 (LinkedIn)
github.com/amitsachdeva45
(Personal)
Top Skills
Core Java
Laravel
Python 3
Languages
Punjabi
Hindi
English
Certifications
Neural Networks and Deep Learning
Generative AI with Large Language
Models
PHP, HTML
Architecting in AWS
Supervised Machine Learning:
Regression and Classification 
Honors-Awards
International Conference on
Contemporary Computing (IC3)
Painting Hub
Emotion Detection using Artificial
Neural Network
National Level Science Talent
Search Examination
Programming Competency Test
Publications
Water disputes between India and
China
Emotion Detection from videos using
Artificial Neural Network
Amit Sachdeva
Software Development Engineer II at Amazon
Vancouver, British Columbia, Canada
Summary
I am currently working as a Software developer in Amazon,
Vancouver. As an industrious and Enthusiast developer, I focus on
good code quality, proper software processes, and development with
goo

In [5]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [7]:
name = "Amit Sachdeva"

In [8]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [9]:
system_prompt

"You are acting as Amit Sachdeva. You are answering questions on Amit Sachdeva's website, particularly questions related to Amit Sachdeva's career, background, skills and experience. Your responsibility is to represent Amit Sachdeva for interactions on the website as faithfully as possible. You are given a summary of Amit Sachdeva's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Amit Sachdeva. I am software development engineer in Amazon. I am from India, and currently citizen in Canada. I moved to Montreal, Canada in 2018. I love keeping myself activity and believe in doing various forms of physical activities including badminton, tennis, gym etc. I love the food specially Indian food, and right now exploring other cuisines like Mexican and mediterranean. Currently staying in Vancouver

In [10]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [11]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [12]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [51]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
You MUST return ONLY a valid JSON object and return format EVALUATION class object. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is_acceptable and your feedback."

In [52]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [53]:
import os
# gemini = OpenAI(
#     api_key=os.getenv("GOOGLE_API_KEY"), 
#     base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
# )

deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
deepseek = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com/v1")
model_name = "deepseek-chat"

In [55]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    # response = gemini.beta.chat.completions.parse(model="gemini-2.0-flash", messages=messages, response_format=Evaluation)
    # return response.choices[0].message.parsed
    response = deepseek.chat.completions.create(model="deepseek-chat", messages=messages, response_format={"type": "json_object"})
    return Evaluation.model_validate_json(response.choices[0].message.content)

In [56]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do I play cricket?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [57]:
reply

"While cricket isn't specifically mentioned in my activities, I do enjoy various forms of physical activities such as badminton and tennis. However, I’m always open to exploring new sports, including cricket! If you have any tips or want to discuss cricket further, feel free to share!"

In [58]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=False, feedback="The response is unacceptable because it completely fails to address the user's question about patents. The user asked 'do you hold a patent?' which is a straightforward question about intellectual property, but the agent responded with irrelevant information about cricket and physical activities. This shows a lack of attention to the user's query and failure to provide accurate information from the provided context. The agent should have acknowledged that the context doesn't mention any patents and either said 'I don't know' or provided relevant information about intellectual property if available.")

In [59]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [62]:
def chat(message, history):
    if "cricket" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [63]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Failed evaluation - retrying
The response is not acceptable because it uses Pig Latin ('otnay erymay uchmay easonsray, Iay otnay oplay iketcray. Iay overlay ariousvay ormsfay ofyay ysicalpay actitivies, ikelay admintonbay, ennisatay, andyay ymgay.'), which is unprofessional and confusing for a potential client or employer. The Agent should have responded in clear, professional English, directly answering the question about cricket while engagingly sharing relevant personal interests from the context, such as badminton, tennis, and gym activities. This response fails to meet the standards of professionalism and clarity expected in this scenario.
